<a href="https://colab.research.google.com/github/gitmystuff/DSChunks/blob/main/Coefficient%20Interpretation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Coefficient Interpretation

In [1]:
# get the data
import pandas as pd

advertising = pd.read_csv('https://raw.githubusercontent.com/gitmystuff/Datasets/refs/heads/main/Advertising.csv', usecols=['TV', 'radio', 'newspaper', 'sales'])
advertising.head()

,TV,radio,newspaper,sales
0,230.1,37.8,69.2,22.1
1,44.5,39.3,45.1,10.4
2,17.2,45.9,69.3,9.3
3,151.5,41.3,58.5,18.5
4,180.8,10.8,58.4,12.9


In [2]:
# train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    advertising.drop('sales', axis=1),
    advertising['sales'],
    test_size=0.25,
    random_state=42)

In [3]:
# create and train the model
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train, y_train)

# test set prediction results
yhat = model.predict(X_test)
print(f'MSE: {mean_squared_error(y_true=y_test, y_pred=yhat)}')
print(f'R-Squared: {r2_score(y_test, yhat)}')

MSE: 2.880023730094193
R-Squared: 0.8935163320163657


In [4]:
# make a prediction
d = {'TV': 232.1, 'radio': 8.6, 'newspaper': 8.7}
d = pd.Series(d)
model.predict(pd.DataFrame([d]))

array([14.99230101])

In [5]:
# view the coefficients and intercept
print(list(zip(X_train, model.coef_)))
print(model.intercept_)

[('TV', 0.045433558624649865), ('radio', 0.19145653561741385), ('newspaper', 0.002568090815700641)]
2.778303460245283


In [6]:
# print predictions (yhat) using model.predict
yhat

array([16.38348211, 20.92434957, 21.61495426, 10.49069997, 22.17690456,
       13.02668085, 21.10309295,  7.31813008, 13.56732111, 15.12238649,
        8.92494113,  6.49924401, 14.30119928,  8.77233515,  9.58665483,
       12.09485291,  8.59621605, 16.25337881, 10.16948105, 18.85753401,
       19.5799036 , 13.15877029, 12.25103735, 21.35141984,  7.69607607,
        5.64686906, 20.79780073, 11.90951247,  9.06581044,  8.37295611,
       12.40815899,  9.89416076, 21.42707658, 12.14236853, 18.28776857,
       20.18114718, 13.99303029, 20.89987736, 10.9313953 ,  4.38721626,
        9.58213448, 12.6170249 ,  9.93851933,  8.06816257, 13.45497849,
        5.25769423,  9.15399537, 14.09552838,  8.71029827, 11.55102817])

### The Formula

y = intercept + coef_0(TV) + coef_1(radio) + coef_2(newspaper)

In [7]:
# print predictions using formula with coefficients
print((model.intercept_ + model.coef_[0]*X_test.TV + model.coef_[1]*X_test.radio + model.coef_[2]*X_test.newspaper).tolist())

[16.38348211331145, 20.92434956860307, 21.61495426261631, 10.490699965305927, 22.17690456111917, 13.026680845553397, 21.103092953860898, 7.318130075648669, 13.5673211091523, 15.122386487507857, 8.924941132904676, 6.499244005196173, 14.301199279996299, 8.772335145446654, 9.586654828878146, 12.094852912689179, 8.596216048824564, 16.25337881394479, 10.16948104922519, 18.85753400618875, 19.579903603569882, 13.158770291583766, 12.251037354070593, 21.3514198351807, 7.696076068322204, 5.646869061354872, 20.797800727727505, 11.909512469431657, 9.065810444726882, 8.37295611219897, 12.408158991348456, 9.89416075915536, 21.427076579130425, 12.14236852640649, 18.287768571447476, 20.18114717763739, 13.993030287767779, 20.89987735685254, 10.931395295035014, 4.38721625929602, 9.582134482801438, 12.617024899270742, 9.938519325583735, 8.068162573267706, 13.454978494373576, 5.2576942347480085, 9.15399537430477, 14.095528381492281, 8.71029826946302, 11.551028165836158]


**Interpreting a coefficient**: \$1000 dollars on radio advertising would be associated with an increase of sales by 0.19 * 1000, or 190 units, given spending stays the same.

In [8]:
# add constant and build model
import statsmodels.api as sm
import statsmodels.formula.api as smf

X_train.insert(0, 'const', 1)
model = sm.OLS(y_train, X_train).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                  sales   R-squared:                       0.897
Model:                            OLS   Adj. R-squared:                  0.895
Method:                 Least Squares   F-statistic:                     422.2
Date:                Thu, 10 Oct 2024   Prob (F-statistic):           1.02e-71
Time:                        21:06:15   Log-Likelihood:                -289.20
No. Observations:                 150   AIC:                             586.4
Df Residuals:                     146   BIC:                             598.4
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.7783      0.375      7.415      0.0

In [9]:
# sanity check for radio
print(0.1915 + .010 * 1.96) # upper bound ci
print(0.1915 - .010 * 1.96) # lower bound ci
print(0.1915 / (.010)) # coef/std err=t

0.2111
0.1719
19.15


In [10]:
import numpy as np

radio_coef = model.params.iloc[2]  # Get the coefficient for radio (index 2)
print(radio_coef)
radio_std_err = model.bse.iloc[2]   # Get the standard error for radio (index 2)
print(radio_std_err)
radio_t = radio_coef / radio_std_err
print(radio_t)  # Get the t-statistic for radio

0.19145653561741383
0.010036706260446077
19.075634042607184


In [11]:
# view the coefficients and intercept
print(model.params)

const        2.778303
TV           0.045434
radio        0.191457
newspaper    0.002568
dtype: float64


In [12]:
print(model.bse) # standard erros

const        0.374696
TV           0.001625
radio        0.010037
newspaper    0.007211
dtype: float64


In [13]:
model.tvalues # etc model.pvalues, model.conf_int()

,0
const,7.414812
TV,27.960294
radio,19.075634
newspaper,0.356147


If we want to see 20 units in sales, given a unit is one million dollars, and we are spending \\$38,900 on radio advertising, how much would we need to spend on TV advertising?

Consider this equation:

$
y = \beta_0 + \beta_1(X_1) + \beta_2(X_2)
$

To see an increase of 20 units in sales knowing that we are spending \\$38,900 on radio advertising, what do we need to spend on TV?

In [14]:
# solve for X1
intercept = model.params['const']
B1 = model.params['TV']
B2 = model.params['radio']
B3 = model.params['newspaper']
X2 = 38.9
X3 = 50.6
print(f'y = {intercept:0.2f} +( {B1:0.2f} * X1) + ({B2:0.2f} * {X2}) + ({B3:0.2f} * {X3})')

y = 2.78 +( 0.05 * X1) + (0.19 * 38.9) + (0.00 * 50.6)
